In [1]:
import os
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed

from IPython.display import display, HTML
from langchain_community.document_loaders import PyPDFLoader
from tqdm.auto import tqdm

# Specify the root directory where PDF files exist
root_directory = "/home/gtechsources/Downloads/VetRef-Books"

# Prefer the cached pickle so PDFs are not reloaded every run
cache_candidates = [
    os.path.join(os.getcwd(), "docs_data.pkl"),
    os.path.join(os.getcwd(), "notebook", "docs_data.pkl"),
]
pickle_path = next((path for path in cache_candidates if os.path.exists(path)), cache_candidates[0])

# Optional visual styling for notebook output
display(HTML("""
<style>
    .progress-title {
        font-weight: 700;
        font-size: 15px;
        color: #1f6feb;
        margin-bottom: 8px;
    }
    .progress-note {
        color: #6b7280;
        margin-bottom: 12px;
    }
</style>
<div class='progress-title'>Embedding Loader</div>
<div class='progress-note'>Loading documents from cache if available, otherwise scanning PDFs once and saving the cache.</div>
"""))


def process_pdf(pdf_file_path):
    pdf_loader = PyPDFLoader(pdf_file_path)
    return pdf_loader.load()


def load_docs_from_pdfs():
    docs = []

    pdf_files_to_process = []
    for root, dirs, files in os.walk(root_directory):
        pdf_files_to_process.extend(
            [os.path.join(root, file) for file in files if file.lower().endswith(".pdf")]
        )

    pdf_files_to_process.sort()
    total_files = len(pdf_files_to_process)
    print(f"Found {total_files} PDF files.")

    with ThreadPoolExecutor() as executor:
        future_to_pdf = {
            executor.submit(process_pdf, pdf_file_path): pdf_file_path
            for pdf_file_path in pdf_files_to_process
        }

        with tqdm(
            total=total_files,
            desc="Loading PDFs",
            colour="green",
            dynamic_ncols=True,
            bar_format="{desc}: |{bar}| {n_fmt}/{total_fmt} files [{elapsed}<{remaining}, {rate_fmt}]",
        ) as pbar:
            for future in as_completed(future_to_pdf):
                pdf_file_path = future_to_pdf[future]
                try:
                    batch_docs = future.result()
                    docs.extend(batch_docs)
                    pbar.update(1)
                    pbar.set_postfix_str(os.path.basename(pdf_file_path))
                    tqdm.write(f"Loaded {os.path.basename(pdf_file_path)} -> {len(batch_docs)} document(s)")
                except Exception as exc:
                    pbar.update(1)
                    tqdm.write(f"Error loading {os.path.basename(pdf_file_path)}: {exc}")

    return docs


if os.path.exists(pickle_path):
    with open(pickle_path, "rb") as file:
        docs = pickle.load(file)
    print(f"Loaded {len(docs)} documents from cache: {pickle_path}")
else:
    docs = load_docs_from_pdfs()
    with open(pickle_path, "wb") as file:
        pickle.dump(docs, file)
    print(f"Saved cache to: {pickle_path}")

print(f"Done. Loaded {len(docs)} documents total.")


/home/gtechsources/development/venvs/vetrefs-llama-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 8 PDF files.


Loading PDFs: |█▎        | 1/8 files [00:54<06:20, 54.40s/it]

Loaded pet-owner-educational-atlas-surgery.pdf -> 92 document(s)


Loading PDFs: |██▌       | 2/8 files [01:09<03:04, 30.69s/it]

...

Loaded pet-owner-educational-atlas-dogs.pdf -> 80 document(s)


Loading PDFs: |████████▊ | 7/8 files [03:25<00:30, 30.85s/it]

Loaded pet-owner-educational-atlas-dermatology.pdf -> 64 document(s)


Loading PDFs: |██████████| 8/8 files [03:42<00:00, 27.82s/it]

Loaded pet-owner-educational-atlas-parasites.pdf -> 108 document(s)
Saved cache to: /home/gtechsources/development/vetrefs-llama_copy1/notebook/docs_data.pkl
Done. Loaded 657 documents total.


In [2]:
#import necessary libraries
import os
import openai
from langchain.prompts import PromptTemplate
from langchain_pinecone import PineconeVectorStore
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from dotenv import load_dotenv, find_dotenv
from langchain.document_loaders import PyPDFLoader
from concurrent.futures import ThreadPoolExecutor
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from dotenv import load_dotenv, find_dotenv   


In [3]:
# Load environment variables from the .env file
load_dotenv(find_dotenv())

# Access the environment variable
openai.api_key = os.environ['OPENAI_API_KEY']

# Read from os.environ
index_name = os.getenv("PINECONE_INDEX_NAME")
namespace = os.getenv("PINECONE_NAMESPACE")

print(f"Index: {index_name}")
print(f"Namespace: {namespace}")

import datetime
current_date = datetime.datetime.now().date()
if current_date < datetime.date(2023, 9, 2):
    llm_name = "gpt-3.5-turbo-0301"
else:
    llm_name = "gpt-3.5-turbo"
print(llm_name)

llm = ChatOpenAI(model_name = llm_name, temperature=0)


Index: vetrefs-llama-copy1
Namespace: vetref-copy1
gpt-3.5-turbo


/home/gtechsources/development/venvs/vetrefs-llama-env/lib/python3.12/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


# length of 'docs'

In [4]:
print("Total pages are :",len(docs))


Total pages are : 657


# Load docs object

In [5]:
import pickle

with open('docs_data.pkl', 'rb') as file:
    docs = pickle.load(file)

print("Total pages are:", len(docs))


Total pages are: 657


In [6]:
import os
import time
import hashlib

from tqdm.auto import tqdm
from pinecone import Pinecone, ServerlessSpec

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# ==========================================================
# Configuration
# ==========================================================

INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")
NAMESPACE = os.getenv("PINECONE_NAMESPACE")

DOCUMENT_BATCH_SIZE = 100      # Number of original documents to process
UPLOAD_BATCH_SIZE = 25         # Number of chunks to embed/upload per request

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 300

EMBEDDING_MODEL = os.environ["OPENAI_EMBEDDING_MODEL"]
EMBEDDING_DIMENSION = 1536

PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"

# ==========================================================
# Initialize Pinecone
# ==========================================================

print("🚀 Connecting to Pinecone...")

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

existing_indexes = pc.list_indexes().names()

if INDEX_NAME not in existing_indexes:

    print(f"🆕 Creating index '{INDEX_NAME}'")

    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION,
        ),
    )

    print("⏳ Waiting for index to become ready...")

    while True:
        status = pc.describe_index(INDEX_NAME).status

        if status["ready"]:
            break

        time.sleep(2)

    print("✅ Index created successfully.\n")

else:
    print(f"✅ Using existing index '{INDEX_NAME}'\n")

# ==========================================================
# Initialize Embedding Model
# ==========================================================

embedding = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    chunk_size=UPLOAD_BATCH_SIZE,      # Prevent OpenAI token limit errors
    show_progress_bar=True,
)

# ==========================================================
# Initialize Vector Store (ONLY ONCE)
# ==========================================================

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embedding,
    namespace=NAMESPACE,
)

# ==========================================================
# Text Splitter
# ==========================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# ==========================================================
# Split documents into processing batches
# ==========================================================

document_batches = [
    docs[i:i + DOCUMENT_BATCH_SIZE]
    for i in range(0, len(docs), DOCUMENT_BATCH_SIZE)
]

print("=" * 60)
print(f"📄 Total Documents : {len(docs):,}")
print(f"📦 Total Batches   : {len(document_batches):,}")
print("=" * 60)

# ==========================================================
# Upload (Skip Existing Chunks)
# ==========================================================

start_time = time.time()

total_chunks = 0
uploaded_chunks = 0
skipped_chunks = 0

index = pc.Index(INDEX_NAME)

outer_bar = tqdm(
    document_batches,
    desc="📚 Processing Document Batches",
    unit="batch",
)

for batch in outer_bar:

    # Split documents into chunks
    splits = text_splitter.split_documents(batch)

    total_chunks += len(splits)

    outer_bar.set_postfix(
        chunks=len(splits),
        uploaded=uploaded_chunks,
        skipped=skipped_chunks,
    )

    # ------------------------------------------------------
    # Generate deterministic IDs
    # ------------------------------------------------------

    ids = []

    for doc in splits:

        source = doc.metadata.get("source", "")
        page = str(doc.metadata.get("page", ""))

        unique_text = source + page + doc.page_content

        chunk_id = hashlib.sha256(
            unique_text.encode("utf-8")
        ).hexdigest()

        ids.append(chunk_id)

    # ------------------------------------------------------
    # Upload only missing chunks
    # ------------------------------------------------------

    for start in tqdm(
        range(0, len(splits), UPLOAD_BATCH_SIZE),
        leave=False,
        desc="⬆️ Uploading",
        unit="batch",
    ):

        batch_docs = splits[start:start + UPLOAD_BATCH_SIZE]
        batch_ids = ids[start:start + UPLOAD_BATCH_SIZE]

        # Fetch existing vectors
        existing = index.fetch(
            ids=batch_ids,
            namespace=NAMESPACE,
        )

        existing_ids = set(existing.vectors.keys())

        docs_to_upload = []
        ids_to_upload = []

        for doc, chunk_id in zip(batch_docs, batch_ids):

            if chunk_id in existing_ids:
                skipped_chunks += 1
            else:
                docs_to_upload.append(doc)
                ids_to_upload.append(chunk_id)

        if docs_to_upload:

            vectorstore.add_documents(
                documents=docs_to_upload,
                ids=ids_to_upload,
            )

            uploaded_chunks += len(docs_to_upload)

elapsed = time.time() - start_time

# ==========================================================
# Summary
# ==========================================================

print("\n" + "=" * 60)
print("🎉 Upload Complete")
print("=" * 60)
print(f"📄 Documents Processed : {len(docs):,}")
print(f"✂️ Total Chunks        : {total_chunks:,}")
print(f"✅ Uploaded Chunks     : {uploaded_chunks:,}")
print(f"⏭️ Skipped Chunks      : {skipped_chunks:,}")
print(f"📦 Processing Batches  : {len(document_batches):,}")
print(f"📤 Upload Batch Size   : {UPLOAD_BATCH_SIZE}")
print(f"🗂️ Index              : {INDEX_NAME}")
print(f"📁 Namespace          : {NAMESPACE}")
print(f"⏱️ Total Time         : {elapsed:.2f} seconds")
print("=" * 60)

🚀 Connecting to Pinecone...
🆕 Creating index 'vetrefs-llama-copy1'
⏳ Waiting for index to become ready...


/home/gtechsources/development/venvs/vetrefs-llama-env/lib/python3.12/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


✅ Index created successfully.

📄 Total Documents : 657
📦 Total Batches   : 7


📚 Processing Document Batches:   0%|          | 0/7 [00:00<?, ?batch/s, chunks=110, skipped=0, uploaded=0]

100%|██████████| 1/1 [00:00<00:00,  1.04it/s]


100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


100%|██████████| 1/1 [00:00<00:00,  1.21it/s]


100%|██████████| 1/1 [00:00<00:00,  1.33it/s]


📚 Processing Document Batches:  14%|█▍        | 1/7 [00:20<02:01, 20.32s/batch, chunks=179, skipped=0, uploaded=110]

100%|██████████| 1/1 [00:01<00:00,  1.44s/it]


100%|██████████| 1/1 [00:01<00:00,  1.26s/it]


100%|██████████| 1/1 [00:01<00:00,  1.12s/it]


100%|██████████| 1/1 [00:01<00:00,  1.25s/it]


100%|██████████| 1/1 [00:00<00:00,  1.68it/s]


100%|██████████| 1/1 [00:01<00:00,  1.10s/it]


100%|██████████| 1/1 [00:01<00:00,  1.29s/it]


📚 Processing Document Batches:  29%|██▊       | 2/7 [00:43<01:48, 21.74s/batch, chunks=145, skipped=0, uploaded=289]

100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


100%|██████████| 1/1 [00:01<00:0


🎉 Upload Complete
📄 Documents Processed : 657
✂️ Total Chunks        : 873
✅ Uploaded Chunks     : 873
⏭️ Skipped Chunks      : 0
📦 Processing Batches  : 7
📤 Upload Batch Size   : 25
🗂️ Index              : vetrefs-llama-copy1
📁 Namespace          : vetref-copy1
⏱️ Total Time         : 125.60 seconds


# Load VectorDB

In [7]:
# Initialize embedding using OpenAIEmbeddings class
embedding = OpenAIEmbeddings(show_progress_bar=True)
vectordb = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding,
    namespace=namespace,
)


In [8]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()

print(stats)

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'vetref-copy1': {'vector_count': 873}},
 'total_vector_count': 873}


In [9]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()

print("Total vectors:", stats["total_vector_count"])

if NAMESPACE in stats["namespaces"]:
    print(
        f"Vectors in namespace '{NAMESPACE}':",
        stats["namespaces"][NAMESPACE]["vector_count"]
    )
else:
    print(f"Namespace '{NAMESPACE}' is empty.")

Total vectors: 873
Vectors in namespace 'vetref-copy1': 873


In [10]:
question = "what is guinea pig"


# Similarity Search

In [11]:
question = "What are the symptoms of canine parvovirus?"

docs = vectorstore.similarity_search(
    question,
    k=5
)

docs

100%|██████████| 1/1 [00:00<00:00,  1.33it/s]


[Document(page_content='27\nDISEASES OF THE SMALL INTESTINE\nDISEASES OF THE GASTROINTESTINAL TRACT\n21 Canine parvovirus infection\nCanine parvovirus is a highly contagious virus that can affect all dogs, but unvaccinated dogs and puppies \n(<4/uni00A0months) are at the greatest risk. Following fecal–oral transmission, canine parvovirus/uni00A02 (CPV-2) invades and \ndestroys rapidly dividing cells (e.g., intestinal crypt epithelium, bone marrow progenitors), producing severe \ngastrointestinal signs and neutropenia. Viral infection of the intestinal crypt cells leads to cell death and loss of the \nnormal barrier function, resulting in translocation of bacteria. Parvovirus-induced gastrointestinal signs can lead to \ndevastating and life-threatening fluid and electrolyte losses.\nTreatment\n/uni25A0Aggressive intravenous fluid therapy: \ncrystalloids ± colloids (fresh frozen \nplasma, hydroxyethyl starch).\n/uni25A0Antiemetics for vomiting.\n/uni25A0Broad-spectrum antibiotics: penici

In [12]:
question = "What are the symptoms of canine parvovirus?"

results = vectorstore.similarity_search(question, k=5)

for i, doc in enumerate(results, 1):
    print("=" * 80)
    print(f"Result {i}")
    print("-" * 80)
    print("Metadata:", doc.metadata)
    print()
    print(doc.page_content[:1000])  # First 1000 characters

100%|██████████| 1/1 [00:00<00:00,  2.20it/s]


Result 1
--------------------------------------------------------------------------------
Metadata: {'page': 28.0, 'source': '/home/gtechsources/Downloads/VetRef-Books/pet-owner-educational-atlas-digestive-system.pdf'}

27
DISEASES OF THE SMALL INTESTINE
DISEASES OF THE GASTROINTESTINAL TRACT
21 Canine parvovirus infection
Canine parvovirus is a highly contagious virus that can affect all dogs, but unvaccinated dogs and puppies 
(<4/uni00A0months) are at the greatest risk. Following fecal–oral transmission, canine parvovirus/uni00A02 (CPV-2) invades and 
destroys rapidly dividing cells (e.g., intestinal crypt epithelium, bone marrow progenitors), producing severe 
gastrointestinal signs and neutropenia. Viral infection of the intestinal crypt cells leads to cell death and loss of the 
normal barrier function, resulting in translocation of bacteria. Parvovirus-induced gastrointestinal signs can lead to 
devastating and life-threatening fluid and electrolyte losses.
Treatment
/uni25A0Agg

# Build Prompt Template

In [13]:
from langchain.prompts import PromptTemplate

# Build prompt
template = """Use the following pieces of context to answer the question at the end. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer. 
{context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)


# RetrievalQA Chain

In [14]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

template = """
You are a helpful veterinary AI assistant.

Use ONLY the following context to answer the question.
If the answer is not contained in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:
"""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)


qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": QA_CHAIN_PROMPT
    },
)

In [16]:
qa_chain({"query": question})


/home/gtechsources/development/venvs/vetrefs-llama-env/lib/python3.12/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
100%|██████████| 1/1 [00:00<00:00,  1.34it/s]


{'query': 'What are the symptoms of canine parvovirus?',
 'result': 'The symptoms of canine parvovirus include loss of appetite, vomiting, and diarrhea with blood and foul odor. These symptoms are more common in young puppies, especially if they have not been properly vaccinated.',
 'source_documents': [Document(page_content='27\nDISEASES OF THE SMALL INTESTINE\nDISEASES OF THE GASTROINTESTINAL TRACT\n21 Canine parvovirus infection\nCanine parvovirus is a highly contagious virus that can affect all dogs, but unvaccinated dogs and puppies \n(<4/uni00A0months) are at the greatest risk. Following fecal–oral transmission, canine parvovirus/uni00A02 (CPV-2) invades and \ndestroys rapidly dividing cells (e.g., intestinal crypt epithelium, bone marrow progenitors), producing severe \ngastrointestinal signs and neutropenia. Viral infection of the intestinal crypt cells leads to cell death and loss of the \nnormal barrier function, resulting in translocation of bacteria. Parvovirus-induced gast

# Custom Prompt (ConversationalQAChain)

In [15]:
from langchain.prompts.prompt import PromptTemplate
custom_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question. And then give answer according to it.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CUSTOM_QUESTION_PROMPT = PromptTemplate.from_template(custom_template)


# Memory

In [16]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(
    memory_key= "chat_history",
    return_messages= True
)


# ConversationalRetrievalChain

In [17]:
from langchain.chains import ConversationalRetrievalChain
Retriever  = vectordb.as_retriever()

crc_qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=Retriever,
    memory=memory,
    condense_question_prompt=CUSTOM_QUESTION_PROMPT
    
)


In [18]:
crc_qa({"question": question})


/home/gtechsources/development/venvs/vetrefs-llama-env/lib/python3.12/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(
100%|██████████| 1/1 [00:01<00:00,  1.06s/it]


{'question': 'What are the symptoms of canine parvovirus?',
 'chat_history': [HumanMessage(content='What are the symptoms of canine parvovirus?'),
  AIMessage(content="I don't have information on canine parvovirus symptoms in the provided context.")],
 'answer': "I don't have information on canine parvovirus symptoms in the provided context."}

In [19]:
crc_qa({"question": question})


100%|██████████| 1/1 [00:00<00:00,  1.63it/s]


{'question': 'What are the symptoms of canine parvovirus?',
 'chat_history': [HumanMessage(content='What are the symptoms of canine parvovirus?'),
  AIMessage(content="I don't have information on canine parvovirus symptoms in the provided context."),
  HumanMessage(content='What are the symptoms of canine parvovirus?'),
  AIMessage(content="I don't have information on canine parvovirus symptoms in the provided context.")],
 'answer': "I don't have information on canine parvovirus symptoms in the provided context."}